# 🎙️ Piper Vietnamese Voice Training Studio (Google Colab T4 GPU)

Công cụ **Huấn luyện Mô hình Giọng nói Tiếng Việt VITS** siêu tốc cho **Piper TTS Studio**.

### ⚡ Quy Trình Huấn Luyện 1-Click (Siêu Nhanh, Không Cần Biên Dịch):
1. **Chuẩn bị Dữ liệu:** Dùng Whisper.cpp chạy offline trên máy tính để tạo gói `<voice_name>_dataset.zip` trong ứng dụng TTS Studio. Whisper.cpp không cần cài trên Colab; Colab chỉ nhận ZIP để huấn luyện.
2. **Chọn runtime tương thích:** Trong **Runtime** ➔ **Change runtime type**, chọn **Runtime Version 2025.07** (Python 3.11.13), bật **T4 GPU**, rồi nhấn **Save**. Các runtime 2025.10 trở lên dùng Python 3.12 và không có wheel `piper-phonemize` cần cho Piper legacy trong notebook này.
3. **Chạy các ô bên dưới:**
   - Bấm nút **Play (▶️)** ở **Bước 1** (~30 giây cài đặt môi trường).
   - Bấm nút **Play (▶️)** ở **Bước 2** để mở giao diện Web Gradio.
   - Trong giao diện Gradio, tải lên file `<voice_name>_dataset.zip` vừa tải từ TTS Studio. Chờ thông báo kiểm tra dữ liệu thành công, sau đó bấm **Bắt Đầu Huấn Luyện (Train)** rồi xuất file `.onnx` mang về TTS Studio!

In [ ]:
# @title 🚀 Bước 1: Khởi Tạo Môi Trường Piper Training & Tải Checkpoint Gốc Tiếng Việt (~30 giây) { display-mode: "form" }
import os, sys, subprocess, time, urllib.request
from pathlib import Path

print('======================================================================')
print('🎙️ KHỞI TẠO PIPER TRAINING ENGINE CHO GOOGLE COLAB GPU T4...')
print('======================================================================')

def download_with_progress(url, dest_path, desc='Đang tải file'):
    dest = Path(dest_path)
    if dest.exists() and dest.stat().st_size > 1000000:
        print(f'✅ {desc}: Đã có sẵn ({dest.stat().st_size / (1024*1024):.1f} MB)')
        return str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.download_tmp')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urllib.request.urlopen(req) as resp, open(tmp, 'wb') as f:
            total_size = int(resp.headers.get('Content-Length', 0))
            downloaded = 0
            start_time = time.time()
            last_print = 0
            while True:
                chunk = resp.read(2 * 1024 * 1024)
                if not chunk: break
                f.write(chunk)
                downloaded += len(chunk)
                now = time.time()
                if now - last_print > 0.4 or downloaded == total_size:
                    last_print = now
                    elapsed = max(0.1, now - start_time)
                    speed = (downloaded / (1024 * 1024)) / elapsed
                    pct = int((downloaded / total_size) * 100) if total_size > 0 else 0
                    mb_down = downloaded / (1024 * 1024)
                    mb_tot = total_size / (1024 * 1024)
                    sys.stdout.write(f'\r-> {desc}: [{pct}%] ({mb_down:.1f} MB / {mb_tot:.1f} MB) - {speed:.1f} MB/s')
                    sys.stdout.flush()
        print('')
        tmp.rename(dest)
        print(f'✅ {desc}: Tải hoàn tất!')
        return str(dest)
    except Exception as e:
        if tmp.exists(): tmp.unlink()
        print(f'\n⚠️ Lỗi khi tải {desc}: {e}')
        return None

# 1. Gói hệ thống eSpeak-NG
print('\n-> [1/3] Cài đặt eSpeak-NG & thư viện phụ trợ...')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'espeak-ng', 'libespeak-ng-dev', 'ffmpeg'], check=True)

# 2. Thư viện Python cho Piper Training & UI
print('\n-> [2/3] Cài đặt PyTorch Lightning, Piper-Phonemize, Gradio...')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', 'piper-phonemize==1.1.0'], check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f'Không cài được piper-phonemize cho Python {sys.version.split()[0]}. '
        'Piper legacy cần wheel Linux tương thích (Python 3.9–3.11). '
        'Hãy vào Runtime > Change runtime type, chọn Runtime Version 2025.07 (Python 3.11.13), '
        'bật T4 GPU, lưu lại, rồi chạy lại ô Bước 1.'
    ) from e
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'gradio', 'onnx', 'soundfile', 'pandas',
    'pytorch-lightning==1.9.5', 'tensorboard', 'cython>=0.29,<1', 'librosa', 'onnxruntime'
], check=True)

# W&B is optional; a broken preinstalled copy can crash Lightning during import even with logging disabled.
print('-> Tắt Weights & Biases (không cần cho huấn luyện offline)...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'wandb'], check=False, capture_output=True, text=True)
import importlib.util
if importlib.util.find_spec('wandb') is not None:
    raise RuntimeError('Không gỡ được gói wandb bị lỗi. Hãy khởi động lại Colab runtime rồi chạy lại ô Bước 1.')

# 3. Cài đặt piper_train
print('\n-> [3/3] Cài đặt Piper VITS Training Engine (piper_train)...')
if not os.path.exists('piper'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/rhasspy/piper.git'], check=True)

piper_src = os.path.abspath('piper/src/python')
if os.path.exists(piper_src):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], cwd=piper_src, check=True)
    build_script = os.path.join(piper_src, 'build_monotonic_align.sh')
    build_res = subprocess.run(['bash', build_script], cwd=piper_src, capture_output=True, text=True)
    if build_res.returncode != 0:
        raise RuntimeError(
            'Không biên dịch được monotonic_align. Log build:\n'
            + (build_res.stdout or '') + '\n' + (build_res.stderr or '')
        )
    if piper_src not in sys.path:
        sys.path.insert(0, piper_src)
    os.environ['PYTHONPATH'] = f"{piper_src}:{os.environ.get('PYTHONPATH', '')}"
else:
    raise FileNotFoundError(f'Không tìm thấy mã nguồn piper_train: {piper_src}')

# 4. Tải Checkpoint Tiếng Việt gốc để fine-tune siêu tốc
ckpt_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/epoch=4769-step=919580.ckpt'
cfg_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/config.json'
download_with_progress(ckpt_url, 'base_checkpoint/vi_base.ckpt', desc='Mô hình Tiếng Việt Gốc (Base Checkpoint)')
download_with_progress(cfg_url, 'base_checkpoint/config.json', desc='Cấu hình Ngữ âm Tiếng Việt (Config)')
base_checkpoint_path = Path('base_checkpoint/vi_base.ckpt').resolve()
base_config_path = Path('base_checkpoint/config.json').resolve()
if not base_checkpoint_path.is_file() or base_checkpoint_path.stat().st_size < 1_000_000:
    raise FileNotFoundError(f'Checkpoint tiếng Việt chưa tải được hoặc không hợp lệ: {base_checkpoint_path}. Chạy lại ô Bước 1 khi có kết nối mạng.')
if not base_config_path.is_file() or base_config_path.stat().st_size == 0:
    raise FileNotFoundError(f'Config Piper chưa tải được hoặc rỗng: {base_config_path}. Chạy lại ô Bước 1 khi có kết nối mạng.')

ver_res = subprocess.run(
    [sys.executable, '-c', 'import piper_train, piper_phonemize; from piper_train.vits.monotonic_align import maximum_path; print("PIPER_TRAIN_READY")'],
    env={**os.environ, 'PYTHONPATH': piper_src + os.pathsep + os.environ.get('PYTHONPATH', '')},
    capture_output=True, text=True
)
if ver_res.returncode != 0 or 'PIPER_TRAIN_READY' not in ver_res.stdout:
    raise RuntimeError(f'Piper dependencies are not ready in {sys.executable}:\n{ver_res.stderr or ver_res.stdout}')
print('\n======================================================================')
print('🎉 MÔI TRƯỜNG PIPER ĐÃ SẴN SÀNG! HÃY BẤM NÚT PLAY Ở BƯỚC 2 BÊN DƯỚI.')
print('======================================================================\n')


In [ ]:
# @title 🎙️ Bước 2: Khởi Chạy Giao Diện Web Gradio UI Huấn Luyện { display-mode: "form" }
import os, sys, shutil, subprocess, json, zipfile, glob, re, time, traceback, tempfile, wave, queue, threading
from pathlib import Path
import torch
import pandas as pd
import gradio as gr
try:
    import piper_phonemize
except ImportError as e:
    raise RuntimeError('Thiếu piper-phonemize. Hãy chọn Runtime Version 2025.07 (Python 3.11.13), rồi chạy lại ô Bước 1.') from e

PIPER_SRC_DIR = os.path.abspath('piper/src/python')
if os.path.exists(PIPER_SRC_DIR) and PIPER_SRC_DIR not in sys.path:
    sys.path.insert(0, PIPER_SRC_DIR)
os.environ['PYTHONPATH'] = f"{PIPER_SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"

BASE_DIR = Path('./piper_workspace').resolve()
DATASET_DIR = BASE_DIR / 'dataset'
WAVS_DIR = DATASET_DIR / 'wav'
TRAIN_DIR = BASE_DIR / 'training_dir'
OUTPUT_DIR = BASE_DIR / 'output'
for d in [BASE_DIR, DATASET_DIR, WAVS_DIR, TRAIN_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PROCESS = None

def unpack_dataset_zip(zip_file_obj, voice_name_hint):
    staging_dir = None
    try:
        if not zip_file_obj:
            return '❌ Chưa có file ZIP được chọn.', 'Chưa có dữ liệu', [], voice_name_hint
        if isinstance(zip_file_obj, dict):
            zip_path = zip_file_obj.get('path') or zip_file_obj.get('name')
        elif isinstance(zip_file_obj, (str, os.PathLike)):
            zip_path = zip_file_obj
        else:
            zip_path = getattr(zip_file_obj, 'path', None) or getattr(zip_file_obj, 'name', None)
        if not zip_path or not Path(zip_path).is_file():
            return '❌ Không đọc được file ZIP tải lên. Hãy tải file lại rồi thử lại.', 'Lỗi file tải lên', [], voice_name_hint
        zip_path = Path(zip_path)
        if not zipfile.is_zipfile(zip_path):
            return '❌ File tải lên không đúng định dạng ZIP.', 'Lỗi định dạng', [], voice_name_hint

        staging_dir = Path(tempfile.mkdtemp(prefix='dataset_upload_', dir=BASE_DIR))
        staged_wavs = staging_dir / 'wav'
        staged_wavs.mkdir(parents=True)
        meta_file = staging_dir / 'metadata.csv'
        metadata_items = []
        wav_items = {}
        with zipfile.ZipFile(zip_path, 'r') as z:
            for item in z.infolist():
                if item.is_dir():
                    continue
                filename = item.filename.replace('\\\\', '/')
                basename = Path(filename).name
                if basename.lower() == 'metadata.csv':
                    metadata_items.append(item)
                elif basename.lower().endswith('.wav'):
                    clip_id = Path(basename).stem
                    if clip_id in wav_items:
                        raise ValueError(f'ZIP có tên file WAV bị trùng: {basename}')
                    wav_items[clip_id] = (basename, item)
            if len(metadata_items) != 1:
                raise ValueError('ZIP phải chứa đúng một file metadata.csv.')
            with z.open(metadata_items[0]) as src, open(meta_file, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            for basename, item in wav_items.values():
                with z.open(item) as src, open(staged_wavs / basename, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

        rows = []
        clip_ids = set()
        with open(meta_file, 'r', encoding='utf-8-sig') as f:
            for line_number, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                if '|' not in line:
                    raise ValueError(f'metadata.csv dòng {line_number} thiếu dấu | phân cách.')
                clip_id, text = line.split('|', 1)
                clip_id, text = clip_id.strip(), text.strip()
                if not clip_id or not text or Path(clip_id).name != clip_id:
                    raise ValueError(f'metadata.csv dòng {line_number} có ID hoặc văn bản không hợp lệ.')
                if clip_id in clip_ids:
                    raise ValueError(f'metadata.csv có ID bị trùng: {clip_id}')
                clip_ids.add(clip_id)
                rows.append([clip_id, text])
        if not rows:
            raise ValueError('metadata.csv không có câu thoại nào hợp lệ.')

        wav_ids = set(wav_items)
        missing_wavs = clip_ids - wav_ids
        extra_wavs = wav_ids - clip_ids
        if missing_wavs or extra_wavs:
            details = []
            if missing_wavs:
                details.append(f'thiếu WAV cho {len(missing_wavs)} ID metadata')
            if extra_wavs:
                details.append(f'{len(extra_wavs)} WAV không có trong metadata')
            raise ValueError('Dữ liệu trong ZIP không khớp: ' + '; '.join(details) + '.')

        for clip_id in clip_ids:
            wav_path = staged_wavs / wav_items[clip_id][0]
            try:
                with wave.open(str(wav_path), 'rb') as audio:
                    if audio.getnframes() <= 0 or audio.getframerate() != 22050 or audio.getnchannels() != 1 or audio.getsampwidth() != 2:
                        raise ValueError('cần WAV mono 22050 Hz, PCM 16-bit và có âm thanh')
            except (wave.Error, EOFError) as e:
                raise ValueError(f'File {wav_path.name} không phải WAV hợp lệ: {e}') from e
            except ValueError as e:
                raise ValueError(f'File {wav_path.name} không đúng định dạng Piper: {e}') from e

        v_name = voice_name_hint
        if not v_name or v_name == 'giong_moi':
            base_zip_name = zip_path.stem
            if base_zip_name.endswith('_dataset'):
                base_zip_name = base_zip_name[:-len('_dataset')]
            v_name = re.sub(r'[^a-zA-Z0-9_-]', '_', base_zip_name).lower().strip('_') or 'giong_moi'

        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        staging_dir.replace(DATASET_DIR)
        staging_dir = None
        summary = f'✅ ZIP hợp lệ! Tìm thấy {len(wav_items)} WAV mono 22050 Hz và {len(rows)} câu thoại khớp metadata.csv.'
        status = f'Sẵn sàng huấn luyện: {len(rows)} câu thoại ({len(wav_items)} audio clips)'
        return summary, status, rows, v_name
    except Exception as e:
        return f'❌ Không thể nạp dataset: {str(e)}', 'Lỗi xử lý', [], voice_name_hint
    finally:
        if staging_dir is not None and staging_dir.exists():
            shutil.rmtree(staging_dir, ignore_errors=True)

def start_training(voice_name, epochs, batch_size, resume_from_base, progress=gr.Progress()):
    global TRAIN_PROCESS
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'
        
        meta_file = DATASET_DIR / 'metadata.csv'
        if not meta_file.exists() or not list(WAVS_DIR.glob('*.wav')):
            yield '❌ Chưa có dữ liệu! Vui lòng tải file dataset.zip ở trên trước.', 'Chưa có dataset', ''
            return
        
        progress(0.1, desc='Đang tiền xử lý ngữ âm Tiếng Việt (Preprocess with piper-phonemize)...')
        yield '⏳ Đang tiền xử lý ngữ âm...', 'Đang tiền xử lý dataset', ''
        jsonl_path = DATASET_DIR / 'dataset.jsonl'
        
        # Preprocess command with Vietnamese phonemizer
        prep_cmd = [
            sys.executable, '-m', 'piper_train.preprocess',
            '--language', 'vi',
            '--input-dir', str(DATASET_DIR),
            '--output-dir', str(DATASET_DIR),
            '--dataset-format', 'ljspeech',
            '--sample-rate', '22050',
            '--single-speaker'
        ]
        prep_res = subprocess.run(prep_cmd, capture_output=True, text=True, env=os.environ)
        if prep_res.returncode != 0:
            yield f'❌ Lỗi khi tiền xử lý ngữ âm: {prep_res.stderr or prep_res.stdout}', 'Lỗi tiền xử lý', ''
            return
        
        progress(0.3, desc='Đang khởi tạo cấu hình Piper Training...')
        train_output_dir = TRAIN_DIR / v_id
        train_output_dir.mkdir(parents=True, exist_ok=True)
        
        # Base checkpoint check
        base_ckpt = Path('base_checkpoint/vi_base.ckpt').resolve()
        base_checkpoint_epoch = 4769 if resume_from_base else 0
        target_max_epochs = int(epochs) + base_checkpoint_epoch
        
        # Build train command
        train_cmd = [
            sys.executable, '-c',
            'import pathlib, runpy, sys, torch; '
            'torch.serialization.add_safe_globals([pathlib.PosixPath]); '
            'sys.argv = ["piper_train"] + sys.argv[1:]; '
            'runpy.run_module("piper_train", run_name="__main__")',
            '--dataset-dir', str(DATASET_DIR),
            '--default_root_dir', str(train_output_dir),
            '--accelerator', 'gpu' if torch.cuda.is_available() else 'cpu',
            '--devices', '1',
            '--logger', 'True',
            '--batch-size', str(int(batch_size)),
            '--max_epochs', str(target_max_epochs),
            '--checkpoint-epochs', '1',
            '--quality', 'medium'
        ]
        
        if resume_from_base:
            if not base_ckpt.is_file():
                yield f'❌ Không tìm thấy checkpoint fine-tune: {base_ckpt}. Hãy chạy lại ô Bước 1 để tải checkpoint.', 'Thiếu checkpoint', ''
                return
            train_cmd.extend(['--resume_from_checkpoint', str(base_ckpt)])
        
        progress(0.4, desc=f'Đang huấn luyện mạng VITS trên GPU ({epochs} epochs)...')
        print(f"Executing: {' '.join(train_cmd)}")
        
        train_env = os.environ.copy()
        train_env['WANDB_MODE'] = 'disabled'
        train_env['WANDB_DISABLED'] = 'true'
        train_env['PYTHONUNBUFFERED'] = '1'
        proc = subprocess.Popen(
            train_cmd,
            cwd=str(train_output_dir),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=train_env
        )
        TRAIN_PROCESS = proc
        
        logs = []
        current_epoch = None
        epoch_started_at = None
        epoch_durations = []
        training_started_at = time.monotonic()
        last_ui_update = 0.0
        output_queue = queue.Queue()
        
        def read_training_output():
            pending = ''
            try:
                while True:
                    char = proc.stdout.read(1)
                    if not char:
                        break
                    if char in ('\r', '\n'):
                        if pending.strip():
                            output_queue.put(pending.strip())
                        pending = ''
                    else:
                        pending += char
                if pending.strip():
                    output_queue.put(pending.strip())
            finally:
                output_queue.put(None)
        
        output_reader = threading.Thread(target=read_training_output, daemon=True)
        output_reader.start()
        output_done = False
        while not output_done or proc.poll() is None:
            try:
                clean_line = output_queue.get(timeout=1.0)
                if clean_line is None:
                    output_done = True
                else:
                    logs.append(clean_line)
                    if len(logs) > 80:
                        logs.pop(0)
                    match = re.search(r'Epoch\s+(\d+)', clean_line)
                    if match:
                        observed_epoch = int(match.group(1))
                        now = time.monotonic()
                        if current_epoch is not None and observed_epoch != current_epoch and epoch_started_at is not None:
                            duration = now - epoch_started_at
                            if duration > 0:
                                epoch_durations.append(duration)
                                epoch_durations = epoch_durations[-5:]
                        if observed_epoch != current_epoch:
                            current_epoch = observed_epoch
                            epoch_started_at = now
                        completed_epochs = max(0, min(int(epochs), current_epoch - base_checkpoint_epoch))
                        pct = min(0.95, 0.4 + 0.55 * (completed_epochs / max(1, int(epochs))))
                        progress(pct, desc=f'Đang huấn luyện: Epoch {max(1, completed_epochs)}/{epochs}...')
            except queue.Empty:
                pass
            
            now = time.monotonic()
            if now - last_ui_update >= 1.0:
                elapsed_seconds = int(now - training_started_at)
                elapsed_text = f'{elapsed_seconds // 3600:02d}:{(elapsed_seconds % 3600) // 60:02d}:{elapsed_seconds % 60:02d}'
                if current_epoch is None:
                    epoch_text = 'Đang chờ log epoch đầu tiên'
                    eta_text = 'chưa đủ dữ liệu để ước tính'
                else:
                    completed_epochs = max(0, min(int(epochs), current_epoch - base_checkpoint_epoch))
                    epoch_text = f'Epoch {max(1, completed_epochs)}/{epochs}'
                    if epoch_durations:
                        average_epoch_seconds = sum(epoch_durations) / len(epoch_durations)
                        remaining_epochs = max(0, int(epochs) - completed_epochs)
                        current_epoch_remaining = max(0.0, average_epoch_seconds - (now - epoch_started_at))
                        eta_seconds = int(current_epoch_remaining + max(0, remaining_epochs - 1) * average_epoch_seconds)
                        eta_text = f'~{eta_seconds // 3600:02d}:{(eta_seconds % 3600) // 60:02d}:{eta_seconds % 60:02d}'
                    else:
                        eta_text = 'đang thu thập thời lượng epoch'
                live_status = f'⏳ Đang huấn luyện — {epoch_text} | đã chạy {elapsed_text} | ETA {eta_text}'
                live_logs = '\n'.join(logs[-40:]) or 'Đang chờ log từ Piper Training...'
                yield live_status, 'Đang huấn luyện', live_logs
                last_ui_update = now
        
        proc.wait()
        if proc.returncode != 0:
            error_details = '\n'.join(logs[-40:]) or f'Tiến trình thoát với mã {proc.returncode}.'
            yield f'❌ Huấn luyện thất bại (mã thoát {proc.returncode}):\n' + error_details, 'Lỗi huấn luyện', '\n'.join(logs[-40:])
            return
        
        progress(1.0, desc='✅ Huấn luyện hoàn tất thành công!')
        yield f'✅ Đã huấn luyện thành công {epochs} epochs cho giọng "{voice_name}"! Bạn có thể chuyển sang Tab "2. Xuất ONNX" để tải mô hình.', 'Huấn luyện hoàn tất', '\n'.join(logs[-40:])
    except Exception as e:
        yield f'❌ Lỗi huấn luyện: {str(e)}\n{traceback.format_exc()}', 'Lỗi huấn luyện', ''

def export_onnx_model(voice_name, progress=gr.Progress()):
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'
        
        train_dir = TRAIN_DIR / v_id
        # Check the voice-specific root used by new runs and the shared dataset log root used by older runs.
        checkpoint_roots = [train_dir, DATASET_DIR / 'lightning_logs']
        ckpts = sorted({
            checkpoint
            for root in checkpoint_roots
            if root.exists()
            for checkpoint in glob.glob(str(root / '**/*.ckpt'), recursive=True)
        })
        if not ckpts:
            searched = ', '.join(str(root) for root in checkpoint_roots)
            return f'❌ Không tìm thấy checkpoint huấn luyện cho giọng "{voice_name}". Đã kiểm tra: {searched}. Không thể xuất an toàn từ checkpoint gốc vì đó sẽ chỉ tạo lại giọng nền. Hãy kiểm tra runtime Colab còn giữ checkpoint hay không.', None, None
        
        latest_ckpt = max(ckpts, key=os.path.getmtime)
        print(f'Checkpoint được chọn để export: {latest_ckpt}')
        progress(0.4, desc=f'Đang export checkpoint: {os.path.basename(latest_ckpt)}...')
        
        out_onnx = OUTPUT_DIR / f'{v_id}.onnx'
        out_json = OUTPUT_DIR / f'{v_id}.onnx.json'
        
        export_cmd = [
            sys.executable, '-c',
            'import pathlib, runpy, sys, torch; '
            'torch.serialization.add_safe_globals([pathlib.PosixPath]); '
            'sys.argv = ["piper_train.export_onnx"] + sys.argv[1:]; '
            'runpy.run_module("piper_train.export_onnx", run_name="__main__")',
            latest_ckpt,
            str(out_onnx)
        ]
        exp_res = subprocess.run(export_cmd, capture_output=True, text=True, env=os.environ)
        if exp_res.returncode != 0 or not out_onnx.exists():
            return f'❌ Lỗi xuất ONNX: {exp_res.stderr or exp_res.stdout}', None, None
        
        # Ensure config json exists
        cfg_path = train_dir / 'config.json'
        if not cfg_path.exists():
            cfg_path = Path('base_checkpoint/config.json')
        if cfg_path.exists():
            shutil.copy(str(cfg_path), str(out_json))
        
        progress(1.0, desc='✅ Đã xuất mô hình ONNX thành công!')
        return f'✅ Đã xuất thành công từ checkpoint:\n- {latest_ckpt}\n- {out_onnx.name} ({out_onnx.stat().st_size / (1024*1024):.1f} MB)\n- {out_json.name}', str(out_onnx), str(out_json)
    except Exception as e:
        return f'❌ Lỗi khi xuất ONNX: {str(e)}', None, None

# Build Gradio Web Interface
custom_css = '''
.gradio-container { max-width: 960px !important; margin: auto; font-family: system-ui, sans-serif; }
.main-btn { background: linear-gradient(135deg, #f59e0b, #ea580c) !important; color: #020617 !important; font-weight: 700 !important; }
'''

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title='Piper Voice Clone Studio') as demo:
    gr.Markdown(
        '''
        # 🎙️ Piper Vietnamese Voice Training Studio
        **Huấn luyện mạng nơ-ron VITS Tiếng Việt trên GPU T4 siêu tốc (Không cần biên dịch Whisper).**
        '''
    )
    
    with gr.Tabs():
        # TAB 1: DATASET & TRAINING
        with gr.TabItem('1. Nạp Dataset & Bắt Đầu Huấn Luyện (Train)'):
            gr.Markdown('### 📦 Bước 1: Nạp file `<ten_giong>_dataset.zip` đã chuẩn bị offline từ TTS Studio')
            with gr.Row():
                dataset_zip_input = gr.File(label='Tải file <voice_name>_dataset.zip từ TTS Studio', file_types=['.zip'], type='filepath')
            
            voice_name_input = gr.Textbox(label='Tên Giọng / Voice ID', value='giong_moi', placeholder='VD: hoang_nam, lan_anh')
            btn_load_dataset = gr.Button('🔍 Kiểm Tra Dữ Liệu', variant='secondary')
            dataset_status = gr.Markdown('Chờ nạp file dữ liệu...')
            dataset_table = gr.Dataframe(headers=['ID', 'Văn bản (Tiếng Việt)'], label='Xem trước câu thoại', interactive=False)
            
            gr.Markdown('### ⚙️ Bước 2: Cấu hình Huấn Luyện (Fine-tuning)')
            with gr.Row():
                epochs_slider = gr.Slider(minimum=5, maximum=100, value=25, step=5, label='Số Epochs (Khuyên dùng: 20-30)')
                batch_slider = gr.Slider(minimum=4, maximum=32, value=16, step=4, label='Batch Size (GPU T4)')
            
            resume_check = gr.Checkbox(label='Tinh chỉnh từ Checkpoint Tiếng Việt Gốc (Khuyên dùng - converge sau vài phút)', value=True)
            
            btn_start_train = gr.Button('🚀 BẮT ĐẦU HUẤN LUYỆN (TRAIN MODEL)', elem_classes='main-btn', size='lg')
            train_output_status = gr.Textbox(label='Trạng Thái Huấn Luyện', interactive=False)
            train_log_output = gr.Textbox(label='Nhật ký huấn luyện (cập nhật trực tiếp)', lines=16, max_lines=28, interactive=False)
            
            btn_load_dataset.click(
                fn=unpack_dataset_zip,
                inputs=[dataset_zip_input, voice_name_input],
                outputs=[dataset_status, train_output_status, dataset_table, voice_name_input]
            )
            dataset_zip_input.change(
                fn=unpack_dataset_zip,
                inputs=[dataset_zip_input, voice_name_input],
                outputs=[dataset_status, train_output_status, dataset_table, voice_name_input]
            )
            btn_start_train.click(
                fn=start_training,
                inputs=[voice_name_input, epochs_slider, batch_slider, resume_check],
                outputs=[train_output_status, dataset_status, train_log_output]
            )
        
        # TAB 2: EXPORT ONNX
        with gr.TabItem('2. Xuất File Mô Hình (.onnx & .onnx.json)'):
            gr.Markdown(
                '''
                ### ⚡ Xuất mô hình nén ONNX để nạp vào Piper TTS Studio
                Sau khi quá trình huấn luyện hoàn tất (hoặc đạt số epoch mong muốn), nhấn nút bên dưới để xuất 2 file:
                - `<ten_giong>.onnx`: Trọng số nơ-ron mạng VITS
                - `<ten_giong>.onnx.json`: Cấu hình ngữ âm & mẫu âm học
                '''
            )
            btn_export = gr.Button('⚡ XUẤT FILE .ONNX & .ONNX.JSON', elem_classes='main-btn', size='lg')
            export_status = gr.Markdown('Chờ xuất file...')
            with gr.Row():
                file_onnx_out = gr.File(label='File Mô Hình (.onnx)')
                file_json_out = gr.File(label='File Cấu Hình (.onnx.json)')
            
            btn_export.click(
                fn=export_onnx_model,
                inputs=[voice_name_input],
                outputs=[export_status, file_onnx_out, file_json_out]
            )

demo.queue().launch(share=True, debug=False)
